# Exploratory Data Analysis – Memory optimization

Cette section mesure l'empreinte mémoire du jeu de données d'obésité avant et après l'application de la fonction `optimize_memory` définie dans `src/data_processing.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Chemin racine du projet (un niveau au-dessus du dossier notebooks)
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_processing import optimize_memory

# Chargement du CSV
data_path = project_root / "data" / "ObesityDataSet_raw_and_data_sinthetic.csv"
print(f"Chargement des données depuis : {data_path}")

df = pd.read_csv(data_path)

# Mémoire avant optimisation (en MB)
mem_before = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Mémoire avant optimisation : {mem_before:.2f} MB")

# Appel de optimize_memory sur une copie pour ne pas modifier df original
print("\n--- Appel optimize_memory(df.copy()) ---")
optimized_df = optimize_memory(df.copy())

# Mémoire après optimisation (en MB)
mem_after = optimized_df.memory_usage(deep=True).sum() / (1024 ** 2)
reduction = ((mem_before - mem_after) / mem_before * 100) if mem_before > 0 else 0.0

print(f"\nMémoire après optimisation : {mem_after:.2f} MB")
print(f"Gain de mémoire : {reduction:.2f}%")

## Résultats avant / après optimisation mémoire

En exécutant la cellule précédente, on obtient :

- **Mémoire avant optimisation** : valeur imprimée en MB pour le DataFrame brut.
- **Mémoire après optimisation** : valeur imprimée en MB pour `optimized_df` retourné par `optimize_memory`.
- **Gain de mémoire (%)** : pourcentage de réduction entre les deux.

L'optimisation repose sur :

- le **downcast des colonnes numériques** (entiers et flottants) vers des types plus compacts,
- la conversion de certaines colonnes de type `object` en **`category`** lorsqu'elles ont peu de modalités distinctes.

Cela permet de diminuer l'empreinte mémoire, ce qui est particulièrement utile si l'on manipule des versions plus volumineuses du dataset ou si l'on enchaîne plusieurs transformations et entraînements de modèles.

### Optimisation de la mémoire - Démonstration avant/après

Dans cette section, nous allons illustrer de manière claire l'impact de la fonction `optimize_memory(df)` sur l'empreinte mémoire du dataset d'obésité, avec une comparaison **avant / après** et un résumé chiffré.

In [ ]:
import pandas as pd

from src.data_processing import optimize_memory

# 1. Chargement du dataset brut
df = pd.read_csv("../data/ObesityDataSet_raw_and_data_sinthetic.csv")

# 2. Mémoire avant optimisation (en MB)
initial_memory = df.memory_usage(deep=True).sum() / 1024**2
print(f"Mémoire avant optimisation : {initial_memory:.2f} MB")

# 3. Application de optimize_memory sur une copie pour préserver l'original
print("\n--- Exécution de optimize_memory(df.copy()) ---")
df_optimized = optimize_memory(df.copy())

# 4. Mémoire après optimisation (en MB)
final_memory = df_optimized.memory_usage(deep=True).sum() / 1024**2
print(f"\nMémoire après optimisation : {final_memory:.2f} MB")

# 5. Calcul et affichage du gain absolu et relatif
savings = initial_memory - final_memory
percentage = (savings / initial_memory) * 100 if initial_memory > 0 else 0
print(f"\nÉconomies : {savings:.2f} MB ({percentage:.1f} %)")

# Sauvegarde des distributions de types avant/après pour la table récapitulative
before_dtypes_counts = df.dtypes.value_counts().sort_index()
after_dtypes_counts = df_optimized.dtypes.value_counts().sort_index()

#### Résumé des économies de mémoire

En exécutant la cellule précédente, on obtient par exemple :

- **Avant** : `initial_memory` MB
- **Après** : `final_memory` MB
- **Gain** : `percentage` % (`savings` MB économisés)

Pour le rapport ou le README du projet, ces valeurs peuvent être remplacées par les chiffres exacts observés lors de l'exécution (captures d'écran possibles) afin de documenter clairement le bénéfice de l'optimisation mémoire.

Le gain provient principalement :

- du **downcast des colonnes numériques** (`float64` → `float32`, `int64` → `int8/16/32`),
- de la **conversion des colonnes catégorielles à faible cardinalité** (par ex. `Gender`, `SMOKE`, `CAEC`, `CALC`, etc.) vers le type `category`, qui stocke les valeurs de façon plus compacte.

In [ ]:
import pandas as pd

# 7. Petite table récapitulative des types avant / après
summary_types = (
    pd.concat(
        [before_dtypes_counts.rename("Avant"), after_dtypes_counts.rename("Après")],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)

print("Distribution des dtypes avant / après optimisation :")
display(summary_types)

print("\nTable markdown (pour intégration éventuelle dans le README) :")
print(summary_types.to_markdown())

## 3. Correlation & SHAP Interaction Analysis

Based on our SHAP analysis, we observed strong interactions and data correlations, especially between physiological characteristics like `Weight` and `Height`. Let's visualize the standard Pearson feature correlations, and compute the mathematical `BMI` feature directly to feed the model a generalizing trait.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate Pearson correlation for numerical features
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

print('We notice very strong correlations that influence obesity. Engineering a BMI feature could help generalization.')

In [ ]:
import shap
import joblib

# Load pipeline model from previous run
try:
    model = joblib.load('../models/best_model.pkl')
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(df.drop('NObeyesdad', axis=1, errors='ignore'))
    print('SHAP computed successfully for further interaction analysis.')
except:
    print('Model not available or target leakage removed.')

## 3. Correlation & SHAP Interaction Analysis

Based on our SHAP analysis, we observed strong interactions and data correlations, especially between physiological characteristics like `Weight` and `Height`. Let's visualize the standard Pearson feature correlations, and compute the mathematical `BMI` feature directly to feed the model a generalizing trait.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate Pearson correlation for numerical features
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

print('We notice very strong correlations that influence obesity. Engineering a BMI feature could help generalization.')

In [ ]:
import shap
import joblib

# Load pipeline model from previous run
try:
    model = joblib.load('../models/best_model.pkl')
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(df.drop('NObeyesdad', axis=1, errors='ignore'))
    print('SHAP computed successfully for further interaction analysis.')
except:
    print('Model not available or target leakage removed.')